# Spendly — reproduce V7 spending-alert results

The original V7 alert output was lost. This notebook repeats only its frozen
Isolation Forest and collective-rule comparisons. It does **not** train the LSTM
or claim to recover the original artifacts. Record these results as a new
reproduction on reused synthetic R3 development data.

## Run
1. Attach original R3 `manifest.json`, `train.csv/zip`, `validation.csv/zip`, and
   `calibration.csv/zip`. Attach only one dataset version.
2. CPU is sufficient; GPU can be disabled. Keep `SMOKE=False`, `FEATURE_WORKERS=2`.
3. Run All. Progress shows completed users and forest stages.
4. Download the printed `spendly_alert_review_*_results.zip` **and executed notebook**.

No new hyperparameter search or labels used as features. The same V7 architecture,
budgets, causal features and evaluation boundary are preserved. Library differences
can alter results; versions are recorded. Final/shifted cohorts are not loaded.
An incomplete anomaly stage restarts; a completed, verified stage can be reused by
setting `RESUME_DIR` to your own restored alert-review folder. V7/V8/V9 checkpoint
folders are not compatible. The previous full V7 anomaly stage took about42 minutes;
this is context, not a runtime guarantee. Save/download outputs before leaving.

The collective approach is credited to PR #3. Its usefulness must be assessed
against false alerts, per-type support, event recall and daily alert burden.
Alerts are unusual-spending signals, not fraud findings. No app model is activated.


In [ ]:
CODE_HASH='dca4a21e30a396a07b4ef49f83931bdbfdd8c4b81cd613dd30223c381009993e'
import sys, json, hashlib, time, platform, importlib, importlib.metadata, zipfile
from pathlib import Path
from contextlib import contextmanager
from collections import defaultdict
from types import SimpleNamespace
import numpy as np
import pandas as pd
import joblib
RUN_FINAL_HOLDOUTS=False
ALLOWED_COHORTS=('train','validation','calibration')
TIMINGS=defaultdict(float)
ACCESSES=[]
"""Verified local research checkpoints; restore only your own trusted run folders.

Hashes detect corruption and incompatible inputs, not maliciously replaced bundles.
Joblib/model checkpoints must never be loaded from untrusted sources.
"""
import hashlib
import json
import os
from pathlib import Path
import time
import zipfile

import joblib


def recovery_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path=Path(path); temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,sort_keys=True,indent=2,allow_nan=False),encoding='utf-8')
    os.replace(temporary,path)


def seal_checkpoint(directory, identity, names):
    directory=Path(directory)
    atomic_json(directory/'complete.json',dict(identity=identity,
        files={name:recovery_hash(directory/name) for name in names}))


def verify_checkpoint(directory, identity):
    directory=Path(directory); marker=directory/'complete.json'
    if not marker.exists():
        return False
    record=json.loads(marker.read_text(encoding='utf-8'))
    if record['identity']!=identity:
        raise ValueError('Checkpoint identity mismatch; use a new run for changed code/data/configuration')
    for name,expected in record['files'].items():
        relative=Path(name)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Invalid checkpoint path')
        if recovery_hash(directory/relative)!=expected:
            raise ValueError('Checkpoint artifact hash mismatch: '+name)
    return True


def bind_run(directory, identity):
    """Bind before training; refuse old/unbound nonempty experiment directories."""
    directory=Path(directory); path=directory/'run_identity.json'
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8'))!=identity:
            raise ValueError('Run identity mismatch; do not mix datasets, modes or environments')
        return
    if (directory/'experiments').exists() or (directory/'configuration_lock.json').exists():
        raise ValueError('Older run has no recovery identity; preserve it and start a new run')
    atomic_json(path,identity)


def backup_run(directory, label):
    """Atomic partial archive of evidence/checkpoints; excludes feature cache outside run."""
    directory=Path(directory)
    if not label.replace('_','').isalnum():
        raise ValueError('Invalid backup label')
    archive=directory.parent/(directory.name+'_'+label+'.zip')
    temporary=archive.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for path in sorted(directory.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'):
                z.write(path,Path(directory.name)/path.relative_to(directory))
    os.replace(temporary,archive)
    print('PARTIAL RECOVERY BACKUP:',archive,flush=True)
    print('Download/persist this ZIP now; temporary Kaggle storage can disappear.',flush=True)
    return archive


def runner_snapshot(runner):
    """Two-slot atomic state saves keep the previous valid generation on interruption."""
    root=runner.directory/'recovery'; root.mkdir(exist_ok=True)
    pointer=root/'current.json'
    old=json.loads(pointer.read_text())['slot'] if pointer.exists() else 1
    slot=1-old; folder=root/str(slot); folder.mkdir(exist_ok=True)
    (folder/'complete.json').unlink(missing_ok=True)
    joblib.dump(dict(registry=runner.registry,fold_metrics=runner.fold_metrics,
        runs=runner.runs,histories=runner.histories),folder/'state.joblib')
    seal_checkpoint(folder,runner.identity,['state.joblib'])
    atomic_json(pointer,dict(slot=slot))


def restore_runner(runner):
    root=runner.directory/'recovery'; pointer=root/'current.json'
    if not pointer.exists():
        return
    slot=json.loads(pointer.read_text())['slot']
    if slot not in (0,1):
        raise ValueError('Invalid recovery slot')
    folder=root/str(slot)
    if not verify_checkpoint(folder,runner.identity):
        raise ValueError('Incomplete published runner snapshot')
    saved=joblib.load(folder/'state.joblib')
    for name in ('registry','fold_metrics','runs','histories'):
        setattr(runner,name,saved[name])
    print(f'RESUMED: {len(runner.runs)} completed candidate/seed/fraction experiments',flush=True)


def seal_outputs(directory, names):
    return {name:recovery_hash(Path(directory)/name) for name in names}


def verify_outputs(directory, record):
    for name,expected in record.items():
        if recovery_hash(Path(directory)/name)!=expected:
            raise ValueError('Published evidence changed: '+name)


def checkpoint_stage(runner, name, operation, outputs, inputs=None):
    """Skip a completed stage only after verifying all its named outputs."""
    marker=runner.directory/('stage_'+name); marker.mkdir(exist_ok=True)
    identity=dict(run=runner.identity,inputs=inputs)
    if verify_checkpoint(marker,identity):
        record=json.loads((marker/'outputs.json').read_text())
        for filename,expected in record.items():
            if recovery_hash(runner.directory/filename)!=expected:
                raise ValueError('Stage output changed: '+filename)
        print('RESUMED completed stage:',name,flush=True)
        return
    atomic_json(marker/'status.json',dict(state='running',started=time.time()))
    try:
        operation()
        atomic_json(marker/'outputs.json',{f:recovery_hash(runner.directory/f) for f in outputs})
        seal_checkpoint(marker,identity,['outputs.json'])
        atomic_json(marker/'status.json',dict(state='completed'))
    except Exception as error:
        atomic_json(marker/'status.json',dict(state='failed',error_type=type(error).__name__))
        raise

def safe_json(value):
    if isinstance(value, dict):
        return {str(k):safe_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple,np.ndarray)):
        return [safe_json(v) for v in value]
    if isinstance(value,(np.integer,np.bool_)):
        return value.item()
    if isinstance(value,(float,np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value,(Path,pd.Timestamp)):
        return str(value)
    return value

def write_json(path, value):
    Path(path).write_text(json.dumps(safe_json(value),indent=2,sort_keys=True,allow_nan=False),encoding='utf-8')

def digest(value):
    return hashlib.sha256(json.dumps(safe_json(value),sort_keys=True).encode()).hexdigest()

@contextmanager
def timed(stage):
    start=time.perf_counter()
    try:
        yield
    finally:
        TIMINGS[stage]+=time.perf_counter()-start

def assert_permitted(cohort):
    if cohort not in ALLOWED_COHORTS:
        raise PermissionError('Development-only notebook: requested cohort is prohibited')
    if RUN_FINAL_HOLDOUTS is not False:
        raise PermissionError('Final holdouts are locked; this notebook has no final evaluator')

def discover(name, root):
    # Whitelist exact filenames. Never recursively open arbitrary CSVs/archives.
    if name not in ('manifest.json', *[c+s for c in ALLOWED_COHORTS for s in ('.csv','.zip')]):
        raise PermissionError('Input name is not permitted')
    matches=sorted(Path(root).rglob(name))
    if len(matches)!=1:
        raise FileNotFoundError(f'Attach exactly one {name} under Kaggle Input; found {len(matches)}')
    return matches[0]

def load_manifest(root):
    # Dataset manifest contains aggregate provenance, not labels. Retain permitted entries only.
    raw=json.loads(discover('manifest.json',root).read_text())
    if raw.get('version') not in ('spendly-synthetic-r3-v1','spendly-synthetic-v7-dev-v1') or raw.get('status')!='complete':
        raise ValueError('Expected completed synthetic R3 or V7 development manifest')
    if raw.get('version')=='spendly-synthetic-v7-dev-v1' and set(raw.get('cohorts',{}))!=set(ALLOWED_COHORTS):
        raise ValueError('V7 development manifest must contain only permitted cohorts')
    return dict(version=raw['version'],currency=raw['currency'],timezone=raw['timezone'],
        benchmark_role='fresh synthetic development replication' if raw['version']=='spendly-synthetic-v7-dev-v1' else 'legacy reused development benchmark',
        weeks_per_user=raw['weeks_per_user'],cohorts={k:raw['cohorts'][k] for k in ALLOWED_COHORTS})

def load_cohort(cohort, root, manifest, worker, result_dir):
    assert_permitted(cohort)
    if cohort=='validation' and not (result_dir/'configuration_lock.json').exists():
        raise PermissionError('Legacy validation cannot open until configuration is locked')
    with timed('data_loading'):
        try:
            path=discover(cohort+'.csv',root)
        except FileNotFoundError:
            path=discover(cohort+'.zip',root)
        if path.suffix=='.zip':
            with zipfile.ZipFile(path) as z:
                if z.namelist()!=[cohort+'.csv']:
                    raise ValueError('Attach a single-cohort ZIP containing only '+cohort+'.csv')
                contents=z.read(cohort+'.csv')
        else:
            contents=path.read_bytes()
        if hashlib.sha256(contents).hexdigest()!=manifest['cohorts'][cohort]['sha256']:
            raise ValueError('R3 data hash mismatch')
        import io
        raw=pd.read_csv(io.BytesIO(contents),dtype={'user_id':str,'transaction_id':str})
        if len(raw)!=manifest['cohorts'][cohort]['rows'] or raw.user_id.nunique()!=manifest['cohorts'][cohort]['users']:
            raise ValueError('Permitted cohort composition mismatch')
        data=worker.clean_data(raw)
        ACCESSES.append(dict(cohort=cohort,sha256=manifest['cohorts'][cohort]['sha256'],raw_rows=len(raw),
            users=data.user_id.nunique(),expense_rows=len(data),income_rows=int(raw.transaction_type.eq('income').sum()),
            missing_cells=int(raw.isna().sum().sum()),anomaly_prevalence=float(data.label.mean()),
            category_counts=data.category.value_counts().to_dict(),start=str(data.observation_start.min()),end=str(data.observation_end.max())))
        return data

def restrict_history(data, end):
    result=data.loc[data.transaction_timestamp.lt(end)].copy()
    result['observation_end']=end
    return result
def anomaly_pipeline(runner, calibration, validation, ref):
    """Keep V6's selected single IF excess128 and fixed union as comparators.

    No new IF architecture selection on reused validation. Calibration cohort only
    sets the original 1% FPR budget (0.5% per union component).
    """
    assert not set(calibration.user_id)&(set(runner.data.user_id)|set(validation.user_id))
    with timed('anomaly_detection'):
        stage_started=time.perf_counter()
        from joblib import Parallel,delayed,parallel_config
        def build(data,cohort):
            started=time.perf_counter()
            groups=data.groupby('user_id',sort=True); total=groups.ngroups
            print(f'Anomaly {cohort} features: start, 0/{total} users',flush=True)
            parts=[]
            with parallel_config(backend='loky',n_jobs=runner.workers,inner_max_num_threads=1):
                results=Parallel(return_as='generator',pre_dispatch=runner.workers)(
                    delayed(runner.worker.anomaly_bundle)(g) for _,g in groups)
                for count,part in enumerate(results,1):
                    parts.append(part)
                    if count==1 or count%25==0:
                        print(f'Anomaly {cohort} features: {count}/{total} users; elapsed {time.perf_counter()-started:.1f}s',flush=True)
            result=pd.concat([p[0] for p in parts],ignore_index=True),pd.concat([p[1] for p in parts],ignore_index=True)
            print(f'Anomaly {cohort} features: done, {len(parts)}/{total} users; elapsed {time.perf_counter()-started:.1f}s',flush=True)
            return result
        print('CPU: preserved Isolation Forest fitting and calibration',flush=True)
        a,x=build(runner.data,'train'); ca,cx=build(calibration,'calibration'); va,vx=build(validation,'validation')
        weeks=int((runner.data.observation_end.min()-runner.data.observation_start.min())/pd.Timedelta(weeks=1))
        # runner training history is 120 weeks; calibration/validation begin there.
        boundary=runner.data.observation_start.min()+pd.Timedelta(weeks=weeks)
        cm=ref.event_contained(ca,ca.time.ge(boundary)); vm=va.time.ge(boundary)
        settings=dict(trees=12 if runner.smoke else 300,seed=42)
        specs=[('single',dict(view='excess',samples=128),.01),
               ('union_excess',dict(view='excess',samples=128),.005),
               ('union_rhythm',dict(view='rhythm_only',samples=512),.005)]
        fitted=[]; rows=[]; flags={}
        for name,spec,budget in specs:
            started=time.perf_counter()
            print(f'Anomaly forest {name}: fit start',flush=True)
            fit=ref.fit_forest(spec,x,settings)
            print(f'Anomaly forest {name}: fit complete; elapsed {time.perf_counter()-started:.1f}s',flush=True)
            started=time.perf_counter()
            print(f'Anomaly forest {name}: calibration start',flush=True)
            threshold=ref.calibrate_forest(fit,ca.loc[cm],cx.loc[cm],budget)
            print(f'Anomaly forest {name}: calibration complete; elapsed {time.perf_counter()-started:.1f}s',flush=True)
            fit.update(name=name,threshold=threshold); fitted.append(fit)
        # Companion parameters fixed in source. Calibrate using negatives only,
        # before examining companion benchmark outcomes; no sweep on validation.
        calibrated={name:ref.capped_rule_threshold(cx.loc[cm,'collective_score'],ca.loc[cm,'label'],budget)
                    for name,budget in [('collective_rules',.01),('collective_component',.005)]}
        write_json(runner.directory/'collective_protocol.json',dict(**ref.COLLECTIVE_PROTOCOL,
            thresholds=calibrated,benchmark_role=runner.benchmark_role,
            execution_mode='software_smoke' if runner.smoke else 'full_development',
            adopted=False,reason='predeclared diagnostic; requires a future confirmation protocol to promote'))
        for fit in fitted:
            name=fit['name']; threshold=fit['threshold']
            started=time.perf_counter()
            print(f'Anomaly forest {name}: scoring start',flush=True)
            scores=ref.forest_score(fit,vx.loc[vm]); flags[name]=scores>=threshold['threshold']
            print(f'Anomaly forest {name}: scoring complete; elapsed {time.perf_counter()-started:.1f}s',flush=True)
            rows.append(dict(model=name,cohort='validation',benchmark_role=runner.benchmark_role,**ref.classification_metrics(va.loc[vm,'label'],scores,threshold['threshold'])))
        union=flags['union_excess']|flags['union_rhythm']
        union_metrics=ref.classification_metrics(va.loc[vm,'label'],union.astype(float),.5)
        union_metrics.update(AP=np.nan,ROC_AUC=np.nan)
        rows.append(dict(model='two_if_union',cohort='validation',benchmark_role=runner.benchmark_role,**union_metrics))
        flags['two_if_union']=union
        rule_scores=vx.loc[vm,'collective_score'].to_numpy()
        flags['collective_rules']=rule_scores>=calibrated['collective_rules']['threshold']
        flags['forest_collective_union']=flags['union_excess']|(rule_scores>=calibrated['collective_component']['threshold'])
        for name in ('collective_rules','forest_collective_union'):
            measured=ref.classification_metrics(va.loc[vm,'label'],flags[name].astype(float),.5)
            measured.update(AP=np.nan,ROC_AUC=np.nan)
            rows.append(dict(model=name,cohort='validation',benchmark_role=runner.benchmark_role,**measured))
        families=[]; burden=[]
        for name,predicted in flags.items():
            per_type,daily=ref.collective_diagnostics(va.loc[vm],predicted,name)
            families.extend(per_type); burden.append(daily)
        pd.DataFrame(families).to_csv(runner.directory/'anomaly_per_type_events.csv',index=False)
        write_json(runner.directory/'anomaly_alert_burden.json',burden)
        pd.DataFrame(rows).to_csv(runner.directory/'anomaly_metrics.csv',index=False)
        joblib.dump(fitted,runner.directory/'isolation_forests.joblib')
        write_json(runner.directory/'anomaly_protocol.json',dict(selected='single',view='excess',samples=128,
            architecture_source='frozen V6 selected architecture; no V7 IF search',
            calibration='separate calibration users, original FPR-capped method',
            compatibility_correction='architecture locked before legacy benchmark rather than selected on it again',
            thresholds=[dict(name=f['name'],**f['threshold']) for f in fitted]))
        print(f'Anomaly stage complete: outputs written; elapsed {time.perf_counter()-stage_started:.1f}s',flush=True)

In [ ]:
# CPU-only. Keep the original R3 dataset attached; no LSTM training is run.
INPUT_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working')
SMOKE=False
FEATURE_WORKERS=2
RESUME_DIR=None  # restore only your own alert-review result folder
RESULTS=Path(RESUME_DIR) if RESUME_DIR else WORK_ROOT/('spendly_alert_review_'+time.strftime('%Y%m%d_%H%M%S'))
if RESUME_DIR:
    if not (RESULTS/'run_identity.json').is_file():
        raise ValueError('No alert-review recovery identity in this folder')
else:
    RESULTS.mkdir(parents=True,exist_ok=False)


In [ ]:
WORKER_SOURCE='import hashlib, json\n\nfrom pathlib import Path\n\nfrom collections import defaultdict, deque\n\nimport numpy as np\n\nimport pandas as pd\n\nCOLUMN_MAP = {}\n\ndef parse_dates(values):\n    return pd.to_datetime(values, errors=\'raise\', utc=True, format=\'mixed\').dt.tz_convert(\'Africa/Nairobi\').dt.tz_localize(None)\n\ndef clean_data(raw, blank_normal=False):\n    data = raw.rename(columns=COLUMN_MAP).copy()\n    if not data.columns.is_unique:\n        raise ValueError(\'COLUMN_MAP creates duplicate columns\')\n    required_columns = {\'user_id\', \'transaction_timestamp\', \'amount\', \'category\'}\n    missing_columns = required_columns - set(data.columns)\n    if missing_columns:\n        raise ValueError(\'Missing columns: \' + \', \'.join(sorted(missing_columns)))\n    for column in (\'user_id\', \'category\'):\n        if data[column].isna().any() or data[column].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(column + \' contains missing values\')\n        data[column] = data[column].astype(str).str.strip()\n    data[\'transaction_timestamp\'] = parse_dates(data.transaction_timestamp)\n    data[\'amount\'] = pd.to_numeric(data.amount, errors=\'raise\')\n    if data.transaction_timestamp.isna().any() or not np.isfinite(data.amount).all() or data.amount.lt(0).any():\n        raise ValueError(\'Dates must be present and amounts finite and non-negative KES\')\n    if \'transaction_id\' not in data:\n        data[\'transaction_id\'] = [\'row_\' + str(i) for i in range(len(data))]\n    if data.transaction_id.isna().any():\n        raise ValueError(\'transaction_id contains missing values\')\n    data[\'transaction_id\'] = data.transaction_id.astype(str).str.strip()\n    if not data.transaction_id.is_unique or data.transaction_id.eq(\'\').any():\n        raise ValueError(\'transaction_id must be nonblank and unique; resolve duplicate records first\')\n    if \'transaction_type\' not in data:\n        data[\'transaction_type\'] = \'expense\'\n    data[\'transaction_type\'] = data.transaction_type.astype(str).str.strip().str.lower()\n    if not data.transaction_type.isin([\'expense\', \'income\']).all():\n        raise ValueError(\'transaction_type must be expense or income\')\n    labels = data.get(\'is_anomaly\', pd.Series(pd.NA, index=data.index)).astype(\'string\').str.strip().str.lower()\n    mapping = {\'true\': 1., \'1\': 1., \'1.0\': 1., \'yes\': 1., \'false\': 0., \'0\': 0., \'0.0\': 0., \'no\': 0.}\n    if (labels.notna() & labels.ne(\'\') & ~labels.isin(mapping)).any():\n        raise ValueError(\'is_anomaly contains invalid labels; use 0/1, true/false, yes/no or blank\')\n    data[\'label\'] = labels.map(mapping).astype(float)\n    if blank_normal:\n        if \'is_anomaly\' not in data:\n            raise ValueError(\'Cannot infer normal labels when is_anomaly is entirely absent\')\n        data[\'label\'] = data.label.fillna(0.)\n    data[\'merchant\'] = data.get(\'merchant\', pd.Series(\'\', index=data.index)).fillna(\'\').astype(str).str.strip()\n    data[\'event_id\'] = data.get(\'event_id\', pd.Series(pd.NA, index=data.index)).astype(\'string\').str.strip().replace(\'\', pd.NA)\n    coverage = {\'observation_start\', \'observation_end\'} & set(data.columns)\n    if coverage and len(coverage) != 2:\n        raise ValueError(\'Provide both observation_start and exclusive observation_end\')\n    for column in coverage:\n        data[column] = parse_dates(data[column])\n        if data[column].isna().any() or data.groupby(\'user_id\')[column].nunique().ne(1).any():\n            raise ValueError(column + \' must be present and constant per user\')\n    if coverage and ((data.transaction_timestamp < data.observation_start) | (data.transaction_timestamp >= data.observation_end)).any():\n        raise ValueError(\'Transactions fall outside declared observation coverage\')\n    if \'anomaly_type\' in data:\n        data[\'family\'] = data[\'anomaly_type\'].fillna(\'unspecified\').astype(str)\n    expense = data.loc[data.transaction_type.eq(\'expense\')].copy()\n    if expense.empty:\n        raise ValueError(\'No expense records found\')\n    return expense.sort_values([\'user_id\', \'transaction_timestamp\', \'transaction_id\']).reset_index(drop=True)\n\nA_FEATURES = [\'amount_ratio\', \'category_ratio\', \'category_deviation\', \'merchant_ratio\', \'count_hour\', \'sum_hour_ratio\', \'count_day\', \'sum_week_ratio\', \'duplicate_hour\', \'merchants_hour\', \'night_rarity\', \'category_rarity\', \'user_cold\', \'merchant_missing\']\n\ndef next_occurrence(schedule, current):\n    if schedule[\'step\'] == \'month\':\n        month = current.normalize().replace(day=1) + pd.DateOffset(months=1)\n        day = min(schedule[\'anchor\'], month.days_in_month)\n        return month.replace(day=day) + (schedule[\'last\'] - schedule[\'last\'].normalize())\n    return current + pd.Timedelta(days=round(schedule[\'period\']))\n\nV5_F_FEATURES = ([f\'lag{i}\' for i in range(1, 9)] +\n                 [f\'nonrec_lag{i}\' for i in range(1, 9)] +\n                 [\'mean4\', \'std8\', \'recurring\', \'nonrecurring\', \'log_scale\',\n                  \'nonrec_median\', \'nonrec_std\', \'due_confidence\', \'count4\', \'zero_fraction\'])\n\nRHYTHM_FEATURES = [\'day_count_excess\', \'week_count_excess\', \'gap_compression\',\n                   \'recurring_increase\', \'recurring_known\']\n\ndef _spread(values):\n    """Robust MAD-based spread with a one-transaction floor for sparse users."""\n    if not len(values):\n        return 1.\n    array = np.asarray(values, dtype=float)\n    centre = float(np.median(array))\n    return max(float(np.median(np.abs(array - centre))) * 1.4826, 1.)\n\ndef rhythm_features(data):\n    """One forward pass per user; simultaneous transactions share only earlier history."""\n    rows = []\n    for _, group in data.groupby(\'user_id\', sort=False):\n        day_counts, week_counts, gaps = deque(maxlen=90), deque(maxlen=26), deque(maxlen=200)\n        amounts, times = defaultdict(lambda: deque(maxlen=12)), defaultdict(lambda: deque(maxlen=12))\n        recent = deque()\n        current_day = current_week = None\n        day_total = week_total = 0\n        previous = None\n        for t, batch in group.groupby(\'transaction_timestamp\', sort=True):\n            day, week = t.normalize(), t.normalize() - pd.Timedelta(days=t.weekday())\n            if current_day is None:\n                current_day, current_week = day, week\n            # Completed periods only; the in-progress day/week never enters its own baseline.\n            while current_day < day:\n                day_counts.append(day_total)\n                current_day, day_total = current_day + pd.Timedelta(days=1), 0\n            while current_week < week:\n                week_counts.append(week_total)\n                current_week, week_total = current_week + pd.Timedelta(weeks=1), 0\n            while recent and recent[0] < t - pd.Timedelta(days=7):\n                recent.popleft()\n            day_window = sum(1 for r in recent if r >= t - pd.Timedelta(days=1))\n            week_window = len(recent)\n            day_median = float(np.median(day_counts)) if day_counts else 0.\n            week_median = float(np.median(week_counts)) if week_counts else 0.\n            day_excess = max(0., (day_window - day_median) / _spread(day_counts))\n            week_excess = max(0., (week_window - week_median) / _spread(week_counts))\n            gap = (t - previous).total_seconds() / 60 if previous is not None else None\n            compression = 0.\n            if gap is not None and gaps:\n                compression = min(float(np.median(gaps)) / max(gap, 1.), 100.)\n            for row in batch.itertuples():\n                key = (row.category, row.merchant)\n                past_times, past_amounts = times[key], amounts[key]\n                increase, known = 0., 0.\n                if len(past_times) >= 3:\n                    intervals = np.diff(pd.DatetimeIndex(past_times).to_numpy()) / np.timedelta64(1, \'D\')\n                    cadence = float(np.median(intervals))\n                    if 25 <= cadence <= 35 and float(np.median(np.abs(intervals - cadence))) <= .2 * cadence:\n                        expected = float(np.median(past_amounts))\n                        if expected > 0:\n                            known, increase = 1., max(0., row.amount / expected - 1.)\n                rows.append((row.Index, [day_excess, week_excess, compression, increase, known]))\n            for row in batch.itertuples():\n                times[(row.category, row.merchant)].append(t)\n                amounts[(row.category, row.merchant)].append(row.amount)\n                recent.append(t)\n                day_total += 1\n                week_total += 1\n            # Record the gap only after scoring, so it cannot inform its own baseline.\n            if gap is not None:\n                gaps.append(gap)\n            previous = t\n    return pd.DataFrame([values for _, values in rows], index=[index for index, _ in rows],\n                        columns=RHYTHM_FEATURES, dtype=float).reindex(data.index)\n\ndef json_safe(value):\n    if isinstance(value, dict):\n        return {str(k): json_safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_safe(v) for v in value]\n    if isinstance(value, (np.integer, np.bool_)):\n        return value.item()\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    return value\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef json_write(path, value):\n    path=Path(path)\n    content=json.dumps(json_safe(value), indent=2, sort_keys=True, allow_nan=False)\n    temporary=path.with_name(path.name+\'.writing\')\n    temporary.write_text(content,encoding=\'utf-8\')\n    temporary.replace(path)\n\n"""Fast parity-preserving V5 features, embedded in a CPU-only worker module.\n\nNumerical imports, schemas and the reference helpers are provided by the builder.\nThis file never imports TensorFlow or alters labels, lookbacks or cohort membership.\n"""\n\ndef fast_schedule(times, amounts):\n    dates = pd.DatetimeIndex(times).unique().sort_values()\n    if len(dates) < 2:\n        return None\n    months = dates.year * 12 + dates.month\n    gaps = np.diff(dates.to_numpy()) / np.timedelta64(1, \'D\')\n    if months.is_unique and dates.day.max() - dates.day.min() <= 3 and np.median(np.diff(months)) <= 2 and np.max(np.diff(months)) <= 3:\n        step, period, anchor = \'month\', 30., int(round(float(np.median(dates.day))))\n    else:\n        period = float(np.median(gaps))\n        if not 5 <= period <= 16 or np.median(np.abs(gaps-period)) > .12*period:\n            return None\n        step, anchor = \'days\', None\n    if len(np.unique(times)) != len(times):\n        # Preserve the exact reference\'s pandas default sort behavior for unequal\n        # tied timestamps at the last-six cutoff. Do not silently change schedules.\n        recent=pd.DataFrame({\'timestamp\':times,\'amount\':amounts}).sort_values(\'timestamp\').amount.tail(6).to_numpy()\n    else:\n        recent = np.asarray(amounts[-6:])\n    amount = float(np.median(recent))\n    if amount <= 0 or float(np.median(np.abs(recent-amount))) > .30*amount:\n        return None\n    return dict(last=dates[-1],amount=amount,step=step,period=period,anchor=anchor,\n                confidence=min(1.,(len(dates)-1)/2))\n\ndef forecast_features_fast(data, variant=\'pooled\'):\n    """Pre-group/slice arrays once instead of rebuilding pandas groupbys every week."""\n    rows, values, skipped = [], [], 0\n    for user, group in data.groupby(\'user_id\',sort=True):\n        group = group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        start = group.observation_start.iloc[0] if \'observation_start\' in group else group.transaction_timestamp.min().normalize()\n        end = group.observation_end.iloc[0] if \'observation_end\' in group else group.transaction_timestamp.max().normalize()\n        first = start.normalize()-pd.Timedelta(days=start.weekday())\n        if first < start:\n            first += pd.Timedelta(weeks=1)\n        last = end.normalize()-pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first,last,freq=\'7D\')\n        if len(boundaries)<10:\n            skipped += 1\n            continue\n        t=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\')\n        amount=group.amount.to_numpy(dtype=float)\n        weeks=(group.transaction_timestamp.dt.normalize()-pd.to_timedelta(group.transaction_timestamp.dt.weekday,unit=\'D\')).to_numpy(dtype=\'datetime64[ns]\')\n        codes=np.searchsorted(boundaries.to_numpy(),weeks)\n        valid=(codes>=0)&(codes<len(boundaries)-1)&(weeks>=first.to_datetime64())\n        weekly=np.bincount(codes[valid],weights=amount[valid],minlength=len(boundaries)-1)\n        counts=np.bincount(codes[valid],minlength=len(boundaries)-1)\n        categories=[]\n        # Positions correspond to sorted transaction arrays. Labels/IDs never used as inputs.\n        for _, positions in group.reset_index(drop=True).groupby(\'category\',sort=True).indices.items():\n            positions=np.asarray(positions)\n            merchants=[positions[np.flatnonzero(group.merchant.iloc[positions].to_numpy()==merchant)]\n                       for merchant in sorted(group.merchant.iloc[positions].unique())]\n            categories.append((positions,merchants))\n        for i in range(8,len(weekly)):\n            left,right=boundaries[i],boundaries[i+1]\n            lag_start=boundaries[i-8]\n            lower=(left-pd.Timedelta(weeks=26)).to_datetime64()\n            upper=left.to_datetime64()\n            lag=lag_start.to_datetime64()\n            past=weekly[i-8:i]\n            scale=max(float(past.mean()),1.)\n            recurring=np.zeros(len(group),dtype=bool)\n            due,confidence=0.,[]\n            for positions,merchant_groups in categories:\n                cat=positions[(t[positions]>=lower)&(t[positions]<upper)]\n                schedule=fast_schedule(t[cat],amount[cat]) if variant==\'pooled\' else None\n                candidates=[(cat,schedule)] if schedule is not None else []\n                if schedule is None:\n                    for member in merchant_groups:\n                        selected=member[(t[member]>=lower)&(t[member]<upper)]\n                        candidates.append((selected,fast_schedule(t[selected],amount[selected])))\n                for selected,schedule in candidates:\n                    if schedule is None:\n                        continue\n                    date=next_occurrence(schedule,schedule[\'last\'])\n                    if date < left-pd.Timedelta(days=40):\n                        continue\n                    while date < right:\n                        if date >= left:\n                            due += schedule[\'amount\']\n                            confidence.append(schedule[\'confidence\'])\n                        date=next_occurrence(schedule,date)\n                    recurring[selected[t[selected]>=lag]]=True\n            mask=(t>=lag)&(t<upper)&~recurring\n            nonrec=np.bincount(codes[mask]-(i-8),weights=amount[mask],minlength=8)\n            mean=float(nonrec.mean())\n            values.append([*(past[::-1]/scale),*(nonrec[::-1]/scale),past[-4:].mean()/scale,\n                past.std()/scale,due/scale,mean/scale,np.log(scale),np.median(nonrec)/scale,\n                nonrec.std()/scale,float(np.mean(confidence)) if confidence else 0.,float(counts[i-4:i].mean()),np.mean(past==0)])\n            rows.append(dict(user_id=user,time=left,end=right,y=weekly[i],scale=scale,last=past[-1],\n                             mean4=past[-4:].mean(),recurring=due+mean,robust_recurring=due+float(np.median(nonrec)),due=due,nonrec_mean=mean))\n    return pd.DataFrame(rows),pd.DataFrame(values,columns=V5_F_FEATURES),skipped\n\ndef anomaly_features_fast(data):\n    """Same 18 V5 features; integer time windows avoid repeated Timedelta/Pandas work."""\n    values=[]\n    for _, group in data.groupby(\'user_id\',sort=False):\n        group=group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        history=deque(maxlen=100)\n        categories=defaultdict(lambda:deque(maxlen=40))\n        merchants=defaultdict(lambda:deque(maxlen=20))\n        recent,category_counts,seen,nights=deque(),defaultdict(int),0,0\n        rows=list(group.itertuples())\n        times=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\').astype(\'int64\')\n        boundaries=np.r_[0,np.flatnonzero(np.diff(times))+1,len(times)]\n        hour_ns,day_ns=3600*10**9,86400*10**9\n        for left,right in zip(boundaries[:-1],boundaries[1:]):\n            stamp=times[left]\n            hour_of_day=rows[left].transaction_timestamp.hour\n            while recent and recent[0][0] < stamp-7*day_ns:\n                recent.popleft()\n            hour=[r for r in recent if r[0]>=stamp-hour_ns]\n            day=[r for r in recent if r[0]>=stamp-day_ns]\n            for row in rows[left:right]:\n                a,c,m=row.amount,row.category,row.merchant\n                typical=max(float(np.median(history)) if history else 1.,1.)\n                cat=categories[c]\n                center=max(float(np.median(cat)) if cat else typical,1.)\n                spread=max(float(np.median(np.abs(np.asarray(cat)-center)))*1.4826 if cat else 0.,center*.2,1.)\n                mc=max(float(np.median(merchants[m])) if m and merchants[m] else center,1.)\n                feature=[a/typical,a/center,max(0.,(a-center)/spread),a/mc,len(hour),sum(r[1] for r in hour)/typical,\n                    len(day),sum(r[1] for r in recent)/typical,\n                    sum(bool(m) and r[2]==m and r[3]==c and abs(r[1]-a)<.01 for r in hour),\n                    len({r[2] for r in hour if r[2]}),float(hour_of_day<5 or hour_of_day>=23)*(1-(nights+1)/(seen+2)),\n                    1-category_counts[c]/max(1,seen),float(seen<10),float(not m)]\n                values.append((row.Index,feature))\n            for row in rows[left:right]:\n                history.append(row.amount); categories[row.category].append(row.amount)\n                if row.merchant:\n                    merchants[row.merchant].append(row.amount)\n                recent.append((stamp,row.amount,row.merchant,row.category))\n                seen+=1; nights+=int(hour_of_day<5 or hour_of_day>=23); category_counts[row.category]+=1\n    x=pd.DataFrame([v for _,v in values],index=[i for i,_ in values],columns=A_FEATURES,dtype=float).reindex(data.index)\n    x[\'category_excess\']=np.maximum(0.,x.category_ratio-1.)\n    x[\'merchant_excess\']=np.maximum(0.,x.merchant_ratio-1.)\n    x[\'rare_large\']=x.category_rarity*np.log1p(x.category_excess)*(1-x.user_cold)\n    x[\'burst_amount\']=x.count_hour*np.log1p(x.amount_ratio)\n    return x\n\ndef infer_schedule(group, minimum=2):\n    """Infer a monthly or weekly schedule using observed history only.\n\n    Category pooling can recover a bill whose merchant changes. Multiple purchases\n    in one month are not interpreted as a monthly bill. No category names/labels.\n    """\n    dates = pd.DatetimeIndex(group.transaction_timestamp).unique().sort_values()\n    if len(dates) < minimum:\n        return None\n    months = dates.year * 12 + dates.month\n    monthly = months.is_unique and dates.day.max() - dates.day.min() <= 3\n    gaps = np.diff(dates.to_numpy()) / np.timedelta64(1, \'D\')\n    if monthly and np.median(np.diff(months)) <= 2 and np.max(np.diff(months)) <= 3:\n        step, period = \'month\', 30.\n        # Scheduled day comes from history, not the generator\'s rent/payday constants.\n        anchor = int(round(float(np.median(dates.day))))\n    else:\n        period = float(np.median(gaps))\n        if not 5 <= period <= 16 or np.median(np.abs(gaps - period)) > .12 * period:\n            return None\n        step, anchor = \'days\', None\n    amounts = group.sort_values(\'transaction_timestamp\').amount.tail(6)\n    amount = float(amounts.median())\n    if amount <= 0 or float(np.median(np.abs(amounts - amount))) > .30 * amount:\n        return None\n    return dict(last=dates[-1], amount=amount, step=step, period=period, anchor=anchor,\n                confidence=min(1., (len(dates) - 1) / 2))\n\ndef schedule_features(history, left, right, lag_start, pooling=True, minimum=2):\n    due, recurring_ids, confidence = 0., set(), []\n    # Prefer category pooling only if the entire category is demonstrably periodic.\n    groups = []\n    for _, category in history.groupby(\'category\', sort=True):\n        schedule = infer_schedule(category, minimum) if pooling else None\n        if schedule is not None:\n            groups.append((category, schedule))\n        else:\n            groups.extend((merchant, infer_schedule(merchant, minimum))\n                          for _, merchant in category.groupby(\'merchant\', sort=True))\n    for group, schedule in groups:\n        if schedule is None:\n            continue\n        next_due = next_occurrence(schedule, schedule[\'last\'])\n        if next_due < left - pd.Timedelta(days=40):\n            continue\n        while next_due < right:\n            if next_due >= left:\n                due += schedule[\'amount\']\n                confidence.append(schedule[\'confidence\'])\n            next_due = next_occurrence(schedule, next_due)\n        recurring_ids.update(group.loc[group.transaction_timestamp.ge(lag_start), \'transaction_id\'])\n    return due, recurring_ids, float(np.mean(confidence)) if confidence else 0.\n\ndef forecast_features_v5(data, variant=\'pooled\'):\n    rows, values, skipped = [], [], 0\n    for user, group in data.groupby(\'user_id\', sort=True):\n        start = group.observation_start.iloc[0] if \'observation_start\' in group else group.transaction_timestamp.min().normalize()\n        end = group.observation_end.iloc[0] if \'observation_end\' in group else group.transaction_timestamp.max().normalize()\n        first = start.normalize() - pd.Timedelta(days=start.weekday())\n        if first < start:\n            first += pd.Timedelta(weeks=1)\n        last = end.normalize() - pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first, last, freq=\'7D\')\n        if len(boundaries) < 10:\n            skipped += 1\n            continue\n        week = group.transaction_timestamp.dt.normalize() - pd.to_timedelta(group.transaction_timestamp.dt.weekday, unit=\'D\')\n        totals = group.groupby(week).amount.agg([\'sum\', \'count\']).reindex(boundaries[:-1], fill_value=0.)\n        weekly = totals[\'sum\'].to_numpy()\n        for i in range(8, len(weekly)):\n            left, right = boundaries[i], boundaries[i + 1]\n            lag_start = boundaries[i - 8]\n            past = weekly[i - 8:i]\n            scale = max(float(np.mean(past)), 1.)  # stable even with sparse/bill-heavy weeks\n            history = group.loc[group.transaction_timestamp.lt(left) & group.transaction_timestamp.ge(left - pd.Timedelta(weeks=26))]\n            due, ids, confidence = schedule_features(history, left, right, lag_start, pooling=variant == \'pooled\')\n            discretionary = history.loc[history.transaction_timestamp.ge(lag_start) & ~history.transaction_id.isin(ids)]\n            dweek = discretionary.transaction_timestamp.dt.normalize() - pd.to_timedelta(discretionary.transaction_timestamp.dt.weekday, unit=\'D\')\n            nonrec = discretionary.groupby(dweek).amount.sum().reindex(boundaries[i - 8:i], fill_value=0.).to_numpy()\n            baseline_mean, baseline_median = due + float(nonrec.mean()), due + float(np.median(nonrec))\n            values.append([*(past[::-1] / scale), *(nonrec[::-1] / scale), past[-4:].mean() / scale,\n                           past.std() / scale, due / scale, nonrec.mean() / scale, np.log(scale),\n                           np.median(nonrec) / scale, nonrec.std() / scale, confidence,\n                           float(totals[\'count\'].iloc[i - 4:i].mean()), np.mean(past == 0)])\n            rows.append(dict(user_id=user, time=left, end=right, y=weekly[i], scale=scale, last=past[-1],\n                             mean4=past[-4:].mean(), recurring=baseline_mean, robust_recurring=baseline_median,\n                             due=due, nonrec_mean=nonrec.mean()))\n    return pd.DataFrame(rows), pd.DataFrame(values, columns=V5_F_FEATURES), skipped\n\n"""CPU-only V7 features. Reference functions are supplied by the notebook builder."""\nfrom dataclasses import dataclass, asdict, replace\nimport time\n\n\n@dataclass(frozen=True)\nclass ForecastConfig:\n    lookback: int = 8\n    sequence_features: tuple = (\'total\', \'nonrecurring\')\n    context_features: tuple = (\'mean4\', \'std8\', \'recurring\', \'nonrecurring\',\n        \'log_scale\', \'nonrec_median\', \'nonrec_std\', \'due_confidence\', \'count4\', \'zero_fraction\')\n    category_vocabulary: tuple = ()\n    target_transform: str = \'mean_residual\'\n    scale_method: str = \'mean\'\n    units: int = 32\n    l2: float = .001\n    learning_rate: float = .0003\n    batch_size: int = 256\n    max_epochs: int = 120\n    seed: int = 42\n    schedule: str = \'fixed\'\n    monitor: str = \'val_weighted_mae\'\n    patience: int = 10\n    min_delta: float = 1e-4\n    ablation: str = \'full\'\n    context_branch: bool = False\n    clipnorm: float = 1.\n    loss: str = \'mae\'\n\n    def validate(self):\n        assert self.lookback in (8, 13, 26)\n        assert self.target_transform in (\'mean_residual\', \'median_residual\')\n        assert self.scale_method in (\'mean\', \'mad\')\n        assert len(set(self.sequence_features)) == len(self.sequence_features)\n        assert len(set(self.context_features)) == len(self.context_features)\n        assert self.ablation in (\'full\', \'sequence_only\', \'context_only\', \'shuffled\')\n        assert self.schedule in (\'fixed\', \'plateau\')\n        assert self.loss in (\'mae\', \'huber\')\n        assert self.units > 0 and self.batch_size > 0 and self.max_epochs > 0\n        return self\n\n\ndef feature_columns(lookback):\n    return ([f\'lag{i}\' for i in range(1, lookback+1)] +\n            [f\'nonrec_lag{i}\' for i in range(1, lookback+1)] +\n            list(ForecastConfig().context_features))\n\n\ndef make_features(data, config, reference=False):\n    """Causal schedules use [cutoff-26 weeks,cutoff); target is [cutoff,cutoff+7d).\n\n    Nonrecurring lag amounts are classified using information at that forecast\n    cutoff, not the subsequent target. Sequence is ordered oldest -> newest.\n    All alternatives retain the original eight-week baseline for comparisons.\n    """\n    config.validate()\n    fun = forecast_reference_config if reference else forecast_fast_config\n    f, x, skipped = fun(data, config.lookback)\n    if f.empty:\n        raise ValueError(\'Insufficient complete history; no fabricated target windows\')\n    if config.lookback != 8:\n        original, _, _ = fun(data, 8)\n        aligned = original.set_index([\'user_id\',\'time\']).loc[pd.MultiIndex.from_frame(f[[\'user_id\',\'time\']])]\n        for name in (\'recurring\',\'robust_recurring\',\'due\',\'nonrec_mean\',\'last\',\'mean4\'):\n            f[name] = aligned[name].to_numpy()\n    seqs, contexts, diagnostics = [], [], []\n    for uid, positions in f.groupby(\'user_id\', sort=False).indices.items():\n        group = data.loc[data.user_id.eq(uid)]\n        start, end = group.observation_start.iloc[0], group.observation_end.iloc[0]\n        first = start.normalize() + pd.Timedelta(days=(-start.weekday()) % 7)\n        last = end.normalize()-pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first, last, freq=\'7D\')\n        week = group.transaction_timestamp.dt.normalize()-pd.to_timedelta(group.transaction_timestamp.dt.weekday,unit=\'D\')\n        totals = group.groupby(week).amount.sum().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        counts = group.groupby(week).size().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        days = group.assign(day=group.transaction_timestamp.dt.normalize()).groupby(week).day.nunique().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        largest = group.groupby(week).amount.max().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        weekend_mask=group.transaction_timestamp.dt.weekday.ge(5)\n        weekend_totals=group.loc[weekend_mask].groupby(week[weekend_mask]).amount.sum().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        cats = {}\n        for cat in config.category_vocabulary:\n            chosen = ~group.category.isin(config.category_vocabulary[:-1]) if cat == \'OTHER\' else group.category.eq(cat)\n            cats[cat] = group.loc[chosen].groupby(week[chosen]).amount.sum().reindex(boundaries[:-1],fill_value=0.).to_numpy()\n        for pos in positions:\n            row, base = f.iloc[pos], x.iloc[pos]\n            i = boundaries.get_loc(row.time)\n            past = totals[i-config.lookback:i]\n            mean_scale = float(row.scale)\n            median = float(np.median(past))\n            mad = float(np.median(np.abs(past-median))) * 1.4826\n            scale = mean_scale if config.scale_method == \'mean\' else max(mad, median*.1, 1.)\n            total = np.asarray([base[f\'lag{k}\'] for k in range(config.lookback,0,-1)]) * mean_scale\n            nonrec = np.asarray([base[f\'nonrec_lag{k}\'] for k in range(config.lookback,0,-1)]) * mean_scale\n            channels = dict(total=total/scale, nonrecurring=nonrec/scale,\n                            count=counts[i-config.lookback:i], active_days=days[i-config.lookback:i]/7.,\n                            ticket_mean=totals[i-config.lookback:i]/np.maximum(counts[i-config.lookback:i],1.)/scale,\n                            largest_share=largest[i-config.lookback:i]/np.maximum(totals[i-config.lookback:i],1.),\n                            weekend_share=weekend_totals[i-config.lookback:i]/np.maximum(totals[i-config.lookback:i],1.))\n            context = base.to_dict()\n            # Keep V6 context values exactly for control; alternative robust scale is\n            # explicit and affects only normalized monetary quantities.\n            for name in (\'mean4\',\'std8\',\'recurring\',\'nonrecurring\',\'nonrec_median\',\'nonrec_std\'):\n                context[name] *= mean_scale/scale\n            context[\'log_scale\'] = np.log(scale)\n            recent = totals[max(0,i-13):i]\n            context.update(rolling_median=median/scale, rolling_mad=mad/scale,\n                recent_ratio=float(past[-4:].mean()/max(past.mean(),1.)),\n                trend13=float((recent[-4:].mean()-recent[:4].mean())/scale),\n                week_sin=np.sin(2*np.pi*row.time.isocalendar().week/52.1775),\n                week_cos=np.cos(2*np.pi*row.time.isocalendar().week/52.1775))\n            denominator=max(float(totals[max(0,i-13):i].sum()),1.)\n            for cat, values in cats.items():\n                context[\'share_\'+cat] = float(values[max(0,i-13):i].sum()/denominator)\n            seqs.append((pos,np.stack([channels[n] for n in config.sequence_features],axis=-1)))\n            contexts.append((pos,[context[n] for n in config.context_features]))\n            diagnostics.append((pos,dict(volatility=float(past.std()/max(past.mean(),1.)),\n                activity=float(counts[i-config.lookback:i].mean()), zero_fraction=float(np.mean(past==0)),\n                recurring_share=float(row.due/max(row.recurring,1.)), scale=scale,\n                feature_end=row.time-pd.Timedelta(nanoseconds=1))))\n    sequence=np.asarray([v for _,v in sorted(seqs)],dtype=\'float32\')\n    context=np.asarray([v for _,v in sorted(contexts)],dtype=\'float32\')\n    details=pd.DataFrame([v for _,v in sorted(diagnostics)])\n    for col in details:\n        f[col]=details[col].to_numpy()\n    f[\'center\']=f.recurring if config.target_transform==\'mean_residual\' else f.robust_recurring\n    assert (f.feature_end < f.time).all(), \'Target dates overlap feature dates\'\n    assert (f.end == f.time+pd.Timedelta(weeks=1)).all()\n    assert np.isfinite(sequence).all() and np.isfinite(context).all()\n    return f, sequence, context\n\n\ndef cached_user(raw, config, cache_dir, code_hash):\n    """Only self-created numeric NPZ/CSV caches; configuration and content verified."""\n    feature_keys=(\'lookback\',\'sequence_features\',\'context_features\',\'category_vocabulary\',\'target_transform\',\'scale_method\')\n    metadata=dict(config={k:asdict(config)[k] for k in feature_keys}, code=code_hash, numpy=np.__version__, pandas=pd.__version__,\n        data=hashlib.sha256(pd.util.hash_pandas_object(raw,index=True).values.tobytes()).hexdigest())\n    metadata=json.loads(json.dumps(metadata,sort_keys=True))\n    key=hashlib.sha256(json.dumps(metadata,sort_keys=True).encode()).hexdigest()\n    folder=Path(cache_dir)/key\n    marker=folder/\'complete.json\'\n    if marker.exists():\n        saved=json.loads(marker.read_text())\n        if saved[\'metadata\'] != metadata:\n            raise ValueError(\'Cached metadata differs from active feature configuration\')\n        for name,digest in saved[\'hashes\'].items():\n            if hashlib.sha256((folder/name).read_bytes()).hexdigest()!=digest:\n                raise ValueError(\'Corrupt feature cache\')\n    else:\n        f,s,c=make_features(raw,config)\n        folder.mkdir(parents=True,exist_ok=True)\n        f.to_csv(folder/\'rows.csv\',index=False)\n        np.savez_compressed(folder/\'features.npz\',sequence=s,context=c)\n        hashes={n:hashlib.sha256((folder/n).read_bytes()).hexdigest() for n in (\'rows.csv\',\'features.npz\')}\n        marker.write_text(json.dumps(dict(metadata=metadata,hashes=hashes),sort_keys=True))\n    f=pd.read_csv(folder/\'rows.csv\',dtype={\'user_id\':str},parse_dates=[\'time\',\'end\',\'feature_end\'])\n    with np.load(folder/\'features.npz\',allow_pickle=False) as arrays:\n        return f,arrays[\'sequence\'],arrays[\'context\']\n\n\ndef check_causal_parity(data):\n    """Deterministic sample, all lookbacks, unequal ties, future amount/category edits."""\n    sample=data.loc[data.user_id.eq(sorted(data.user_id.unique())[0])].copy()\n    for lookback in (8,13,26):\n        cfg=ForecastConfig(lookback=lookback)\n        f,s,c=make_features(sample,cfg)\n        rf,rs,rc=make_features(sample,cfg,reference=True)\n        pd.testing.assert_frame_equal(f,rf,check_exact=False,rtol=1e-9,atol=1e-7)\n        np.testing.assert_allclose(s,rs,rtol=1e-6,atol=1e-6)\n        np.testing.assert_allclose(c,rc,rtol=1e-6,atol=1e-6)\n        cutoff=f.time.iloc[len(f)//2]\n        edited=sample.copy()\n        future=edited.transaction_timestamp.ge(cutoff)\n        edited.loc[future,\'amount\']=edited.loc[future,\'amount\']*17+123\n        edited.loc[future,\'category\']=\'FUTURE_ONLY\'\n        ef,es,ec=make_features(edited,cfg)\n        before=f.time.le(cutoff).to_numpy()\n        np.testing.assert_allclose(s[before],es[before],rtol=0,atol=0)\n        np.testing.assert_allclose(c[before],ec[before],rtol=0,atol=0)\n        np.testing.assert_allclose(f.loc[before,\'center\'],ef.loc[before,\'center\'],rtol=0,atol=0)\n    return dict(status=\'passed\',lookbacks=[8,13,26],float32_tolerance=1e-6,\n                future_invariance=\'exact\',reference=\'original pandas implementation generalized only for lookback\')\n\n\n"""Causal companion experiment inspired by BenjaminKakai\'s Spendly PR #3.\n\nIndependent implementation: completed-history baselines, no test-label sweep,\nsame-merchant/category duplicate windows, and calibrated alert budgets.\nThese research scores describe unusual spending, not fraud or confirmed duplicates.\n"""\nfrom collections import deque\n\nimport numpy as np\nimport pandas as pd\n\n\nCOLLECTIVE_PROTOCOL = dict(\n    source=\'https://github.com/Eeshan-Vaghjiani/Spendly/pull/3\',\n    baseline_days=90, warmup_days=28, minimum_weekend_days=4,\n    duplicate_hours=3, duplicate_tolerance=.02,\n    frequency_divisor=4., weekend_divisor=6.,\n    rules_budget=.01, combined_component_budget=.005,\n    selection=\'diagnostic comparator only; retain the V6 single forest\',\n)\n\n\ndef collective_features(data):\n    """Score each timestamp batch before updating history; return original row order.\n\n    The current transaction contributes to its own day-to-date total; simultaneous\n    peers do not see each other. Completed observed zero days enter rate baselines.\n    Matching duplicates must have a nonempty merchant and category, identical for\n    both transactions. No labels, event IDs or future totals inform these features.\n    """\n    required={\'user_id\',\'transaction_id\',\'transaction_timestamp\',\'amount\',\'merchant\',\n              \'category\',\'observation_start\'}\n    if not required <= set(data):\n        raise ValueError(\'Missing collective feature columns\')\n    if data.transaction_id.duplicated().any():\n        raise ValueError(\'Transaction IDs must be unique\')\n    amounts=pd.to_numeric(data.amount,errors=\'coerce\')\n    if not np.isfinite(amounts).all() or (amounts<=0).any():\n        raise ValueError(\'Expected finite positive expense amounts\')\n    work=data.copy().reset_index(drop=True)\n    work[\'_position\']=np.arange(len(work))\n    work[\'transaction_timestamp\']=pd.to_datetime(work.transaction_timestamp)\n    work[\'observation_start\']=pd.to_datetime(work.observation_start)\n    results=[]\n    for _,group in work.groupby(\'user_id\',sort=False):\n        if group.observation_start.nunique()!=1:\n            raise ValueError(\'Inconsistent observation start\')\n        start=group.observation_start.iloc[0]\n        if (group.transaction_timestamp<start).any():\n            raise ValueError(\'Transaction precedes observation\')\n        day=start.normalize()\n        counts,weekends,recent=deque(maxlen=90),deque(maxlen=26),deque()\n        day_count=0; day_amount=0.\n        for stamp,batch in group.groupby(\'transaction_timestamp\',sort=True):\n            current=stamp.normalize()\n            while day<current:\n                # Partial observation-start day is not a complete baseline day.\n                if day>=start:\n                    counts.append(day_count)\n                    if day.weekday()>=5:\n                        weekends.append(day_amount)\n                day+=pd.Timedelta(days=1); day_count=0; day_amount=0.\n            while recent and recent[0][0]<stamp-pd.Timedelta(hours=3):\n                recent.popleft()\n            eligible=len(counts)>=28\n            rate=max(float(np.mean(counts)),1.) if counts else 1.\n            weekend_base=max(float(np.median(weekends)),1.) if weekends else 1.\n            for row in batch.to_dict(\'records\'):\n                merchant=row[\'merchant\']; category=row[\'category\']; amount=float(row[\'amount\'])\n                known=(pd.notna(merchant) and pd.notna(category)\n                       and bool(str(merchant).strip()) and bool(str(category).strip()))\n                matches=sum(1 for _,a,m,c in recent if known and m==merchant and c==category\n                            and abs(a-amount)/max(a,amount)<=.02)\n                frequency=(day_count+1)/rate if eligible else 0.\n                weekend=((day_amount+amount)/weekend_base\n                         if eligible and current.weekday()>=5 and len(weekends)>=4 else 0.)\n                duplicate=float(matches) if eligible else 0.\n                results.append((row[\'_position\'],frequency,weekend,duplicate,eligible,\n                                max(frequency/4.,weekend/6.,duplicate)))\n            for row in batch.to_dict(\'records\'):\n                recent.append((stamp,float(row[\'amount\']),row[\'merchant\'],row[\'category\']))\n                day_count+=1; day_amount+=float(row[\'amount\'])\n    columns=[\'position\',\'day_count_ratio\',\'weekend_ratio\',\'duplicate_count\',\'eligible\',\'collective_score\']\n    out=pd.DataFrame(results,columns=columns).sort_values(\'position\').drop(columns=\'position\')\n    out.index=data.index\n    return out\n\n\ndef capped_rule_threshold(scores, labels, budget):\n    """Tie-safe empirical negative FPR cap, fitted only on calibration labels.\n\n    Positive labels never optimize the threshold. Unknown labels are an error.\n    No negatives means disabled (infinity), rather than uncalibrated alerts.\n    """\n    scores=np.asarray(scores,dtype=float); labels=np.asarray(labels,dtype=float)\n    if scores.shape!=labels.shape or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid calibration arrays\')\n    if not np.isin(labels,[0.,1.]).all() or not 0<budget<1:\n        raise ValueError(\'Explicit binary labels and a fractional budget are required\')\n    negatives=np.sort(scores[labels==0])\n    if not len(negatives):\n        return dict(threshold=float(\'inf\'),negative_support=0,budget=budget,\n                    calibration_FPR=None,status=\'disabled: no labelled negatives\')\n    allowed=int(np.floor(budget*len(negatives)))\n    threshold=max(1.,float(np.nextafter(negatives[len(negatives)-allowed-1],np.inf)))\n    return dict(threshold=threshold,negative_support=len(negatives),budget=budget,\n                calibration_FPR=float(np.mean(negatives>=threshold)),\n                status=\'calibrated\' if allowed else \'calibrated: fewer than one allowed false positive\')\n\n\ndef collective_diagnostics(rows, flags, model):\n    """Descriptive family/event coverage and daily alert burden; never selection."""\n    frame=rows.reset_index(drop=True).copy()\n    flags=np.asarray(flags,dtype=bool)\n    if len(flags)!=len(frame):\n        raise ValueError(\'Prediction alignment mismatch\')\n    frame[\'flag\']=flags\n    frame[\'day\']=pd.to_datetime(frame.time).dt.normalize()\n    records=[]\n    positives=frame.loc[frame.label.eq(1)]\n    for family,part in positives.groupby(\'family\',dropna=False):\n        events=part.dropna(subset=[\'event_id\']).groupby([\'user_id\',\'event_id\']).flag.any()\n        records.append(dict(model=model,family=family,positive_transactions=len(part),\n            transaction_recall=float(part.flag.mean()),events_with_ids=len(events),\n            event_recall=float(events.mean()) if len(events) else None))\n    daily=frame.groupby([\'user_id\',\'day\']).flag.any()\n    return records,dict(model=model,transactions=len(frame),alerts=int(flags.sum()),\n        active_user_days=len(daily),alerted_user_days=int(daily.sum()),\n        fraction_active_days_alerted=float(daily.mean()) if len(daily) else None,\n        event_definition=\'at least one flagged labelled member; absent event IDs excluded\',\n        daily_denominator=\'observed days containing evaluated transactions, not all calendar days\')\n\n\ndef forecast_fast_config(data, lookback, variant=\'pooled\'):\n    """Pre-group/slice arrays once instead of rebuilding pandas groupbys every week."""\n    rows, values, skipped = [], [], 0\n    for user, group in data.groupby(\'user_id\',sort=True):\n        group = group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        start = group.observation_start.iloc[0] if \'observation_start\' in group else group.transaction_timestamp.min().normalize()\n        end = group.observation_end.iloc[0] if \'observation_end\' in group else group.transaction_timestamp.max().normalize()\n        first = start.normalize()-pd.Timedelta(days=start.weekday())\n        if first < start:\n            first += pd.Timedelta(weeks=1)\n        last = end.normalize()-pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first,last,freq=\'7D\')\n        if len(boundaries)<lookback+2:\n            skipped += 1\n            continue\n        t=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\')\n        amount=group.amount.to_numpy(dtype=float)\n        weeks=(group.transaction_timestamp.dt.normalize()-pd.to_timedelta(group.transaction_timestamp.dt.weekday,unit=\'D\')).to_numpy(dtype=\'datetime64[ns]\')\n        codes=np.searchsorted(boundaries.to_numpy(),weeks)\n        valid=(codes>=0)&(codes<len(boundaries)-1)&(weeks>=first.to_datetime64())\n        weekly=np.bincount(codes[valid],weights=amount[valid],minlength=len(boundaries)-1)\n        counts=np.bincount(codes[valid],minlength=len(boundaries)-1)\n        categories=[]\n        # Positions correspond to sorted transaction arrays. Labels/IDs never used as inputs.\n        for _, positions in group.reset_index(drop=True).groupby(\'category\',sort=True).indices.items():\n            positions=np.asarray(positions)\n            merchants=[positions[np.flatnonzero(group.merchant.iloc[positions].to_numpy()==merchant)]\n                       for merchant in sorted(group.merchant.iloc[positions].unique())]\n            categories.append((positions,merchants))\n        for i in range(lookback,len(weekly)):\n            left,right=boundaries[i],boundaries[i+1]\n            lag_start=boundaries[i-lookback]\n            lower=(left-pd.Timedelta(weeks=26)).to_datetime64()\n            upper=left.to_datetime64()\n            lag=lag_start.to_datetime64()\n            past=weekly[i-lookback:i]\n            scale=max(float(past.mean()),1.)\n            recurring=np.zeros(len(group),dtype=bool)\n            due,confidence=0.,[]\n            for positions,merchant_groups in categories:\n                cat=positions[(t[positions]>=lower)&(t[positions]<upper)]\n                schedule=fast_schedule(t[cat],amount[cat]) if variant==\'pooled\' else None\n                candidates=[(cat,schedule)] if schedule is not None else []\n                if schedule is None:\n                    for member in merchant_groups:\n                        selected=member[(t[member]>=lower)&(t[member]<upper)]\n                        candidates.append((selected,fast_schedule(t[selected],amount[selected])))\n                for selected,schedule in candidates:\n                    if schedule is None:\n                        continue\n                    date=next_occurrence(schedule,schedule[\'last\'])\n                    if date < left-pd.Timedelta(days=40):\n                        continue\n                    while date < right:\n                        if date >= left:\n                            due += schedule[\'amount\']\n                            confidence.append(schedule[\'confidence\'])\n                        date=next_occurrence(schedule,date)\n                    recurring[selected[t[selected]>=lag]]=True\n            mask=(t>=lag)&(t<upper)&~recurring\n            nonrec=np.bincount(codes[mask]-(i-lookback),weights=amount[mask],minlength=lookback)\n            mean=float(nonrec.mean())\n            values.append([*(past[::-1]/scale),*(nonrec[::-1]/scale),past[-4:].mean()/scale,\n                past.std()/scale,due/scale,mean/scale,np.log(scale),np.median(nonrec)/scale,\n                nonrec.std()/scale,float(np.mean(confidence)) if confidence else 0.,float(counts[i-4:i].mean()),np.mean(past==0)])\n            rows.append(dict(user_id=user,time=left,end=right,y=weekly[i],scale=scale,last=past[-1],\n                             mean4=past[-4:].mean(),recurring=due+mean,robust_recurring=due+float(np.median(nonrec)),due=due,nonrec_mean=mean))\n    return pd.DataFrame(rows),pd.DataFrame(values,columns=feature_columns(lookback)),skipped\n\ndef forecast_reference_config(data, lookback, variant=\'pooled\'):\n    rows, values, skipped = [], [], 0\n    for user, group in data.groupby(\'user_id\', sort=True):\n        start = group.observation_start.iloc[0] if \'observation_start\' in group else group.transaction_timestamp.min().normalize()\n        end = group.observation_end.iloc[0] if \'observation_end\' in group else group.transaction_timestamp.max().normalize()\n        first = start.normalize() - pd.Timedelta(days=start.weekday())\n        if first < start:\n            first += pd.Timedelta(weeks=1)\n        last = end.normalize() - pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first, last, freq=\'7D\')\n        if len(boundaries) < lookback+2:\n            skipped += 1\n            continue\n        week = group.transaction_timestamp.dt.normalize() - pd.to_timedelta(group.transaction_timestamp.dt.weekday, unit=\'D\')\n        totals = group.groupby(week).amount.agg([\'sum\', \'count\']).reindex(boundaries[:-1], fill_value=0.)\n        weekly = totals[\'sum\'].to_numpy()\n        for i in range(lookback, len(weekly)):\n            left, right = boundaries[i], boundaries[i + 1]\n            lag_start = boundaries[i - lookback]\n            past = weekly[i - lookback:i]\n            scale = max(float(np.mean(past)), 1.)  # stable even with sparse/bill-heavy weeks\n            history = group.loc[group.transaction_timestamp.lt(left) & group.transaction_timestamp.ge(left - pd.Timedelta(weeks=26))]\n            due, ids, confidence = schedule_features(history, left, right, lag_start, pooling=variant == \'pooled\')\n            discretionary = history.loc[history.transaction_timestamp.ge(lag_start) & ~history.transaction_id.isin(ids)]\n            dweek = discretionary.transaction_timestamp.dt.normalize() - pd.to_timedelta(discretionary.transaction_timestamp.dt.weekday, unit=\'D\')\n            nonrec = discretionary.groupby(dweek).amount.sum().reindex(boundaries[i - lookback:i], fill_value=0.).to_numpy()\n            baseline_mean, baseline_median = due + float(nonrec.mean()), due + float(np.median(nonrec))\n            values.append([*(past[::-1] / scale), *(nonrec[::-1] / scale), past[-4:].mean() / scale,\n                           past.std() / scale, due / scale, nonrec.mean() / scale, np.log(scale),\n                           np.median(nonrec) / scale, nonrec.std() / scale, confidence,\n                           float(totals[\'count\'].iloc[i - 4:i].mean()), np.mean(past == 0)])\n            rows.append(dict(user_id=user, time=left, end=right, y=weekly[i], scale=scale, last=past[-1],\n                             mean4=past[-4:].mean(), recurring=baseline_mean, robust_recurring=baseline_median,\n                             due=due, nonrec_mean=nonrec.mean()))\n    return pd.DataFrame(rows), pd.DataFrame(values, columns=feature_columns(lookback)), skipped\n\ndef anomaly_bundle(data):\n    x=anomaly_features_fast(data).join(rhythm_features(data)).join(collective_features(data))\n    a=data[[\'user_id\',\'transaction_id\',\'transaction_timestamp\',\'label\',\'event_id\',\'family\']].rename(columns={\'transaction_timestamp\':\'time\'})\n    return a,x\n'
IF_SOURCE='import numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import IsolationForest\nfrom sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score\nGOAL=.80\ndef rhythm_views(x):\n    """Existing views plus the two declared candidate views."""\n    views = dict(all=[c for c in x.columns if c not in RHYTHM_FEATURES],\n                 context=[\'category_ratio\', \'category_deviation\', \'merchant_ratio\',\n                          \'count_hour\', \'duplicate_hour\', \'rare_large\', \'burst_amount\'],\n                 excess=[\'category_excess\', \'merchant_excess\', \'duplicate_hour\', \'rare_large\', \'burst_amount\'])\n    views[\'rhythm\'] = views[\'excess\'] + RHYTHM_FEATURES\n    views[\'rhythm_only\'] = list(RHYTHM_FEATURES)\n    return views\nRHYTHM_FEATURES=[\'day_count_excess\',\'week_count_excess\',\'gap_compression\',\'recurring_increase\',\'recurring_known\']\n\n\ndef classification_metrics(labels, scores, threshold):\n    labels, scores = np.asarray(labels, float), np.asarray(scores, float)\n    known = np.isfinite(labels)\n    y, alerts = labels[known].astype(bool), scores[known] >= threshold\n    tp, fp, tn, fn = int((y & alerts).sum()), int((~y & alerts).sum()), int((~y & ~alerts).sum()), int((y & ~alerts).sum())\n    both = y.any() and (~y).any()\n    p = tp / (tp + fp) if tp + fp else (0. if len(y) else np.nan)\n    r = tp / (tp + fn) if tp + fn else np.nan\n    f1 = 2 * tp / (2 * tp + fp + fn) if y.any() else np.nan\n    result = dict(rows=len(labels), labelled_rows=len(y), unknown_labels=int((~known).sum()), positive_support=int(y.sum()),\n        accuracy=(tp + tn) / len(y) if len(y) else np.nan, balanced_accuracy=(r + tn / (tn + fp)) / 2 if both else np.nan,\n        precision=p, recall=r, F1=f1, AP=float(average_precision_score(y, scores[known])) if both else np.nan,\n        ROC_AUC=float(roc_auc_score(y, scores[known])) if both else np.nan, FPR=fp / (fp + tn) if fp + tn else np.nan,\n        TP=tp, FP=fp, TN=tn, FN=fn, alert_rate=float(np.mean(scores >= threshold)) if len(scores) else np.nan)\n    result[\'negative_support\'] = int((~y).sum())\n    result[\'metric_status\'] = \'both labelled classes; metrics conditional on label coverage\' if both else \'incomplete label classes: precision/F1/accuracy unavailable\'\n    if not both:\n        for key in (\'precision\', \'F1\', \'accuracy\'):\n            result[key] = np.nan\n    result[\'all_80_targets_met\'] = bool(both and all(result[k] >= GOAL for k in (\'accuracy\', \'balanced_accuracy\', \'precision\', \'recall\', \'F1\')))\n    return result\n\ndef false_positive_floor(labels, scores, max_false_positive_rate):\n    """Most permissive threshold whose calibration false-positive rate stays in budget.\n\n    The release gate constrains false positives among negatives, not total alerts.\n    At 3% prevalence those differ by roughly an order of magnitude, so constraining\n    alert rate instead needlessly caps recall.\n    """\n    labels = np.asarray(labels, dtype=float)\n    scores = np.asarray(scores, dtype=float)\n    if not 0 < max_false_positive_rate < 1:\n        raise ValueError(\'False-positive budget must be a fraction strictly between 0 and 1\')\n    negatives = scores[labels == 0]\n    if not len(negatives):\n        raise ValueError(\'A false-positive budget requires labelled negatives\')\n    ordered = np.sort(negatives)\n    budget = int(np.floor(max_false_positive_rate * len(ordered)))\n    if budget == 0:\n        return float(np.nextafter(ordered[-1], np.inf))\n    unique = np.unique(ordered)\n    false_alerts = len(ordered) - np.searchsorted(ordered, unique, side=\'left\')\n    affordable = unique[false_alerts <= budget]\n    if not len(affordable):\n        return float(np.nextafter(ordered[-1], np.inf))\n    return float(affordable[0])\n\ndef fpr_capped_threshold(labels, scores, fallback, max_false_positive_rate, minimum_positives=None):\n    """Maximise recall subject to the false-positive budget the gate actually imposes."""\n    labels = np.asarray(labels, dtype=float)\n    scores = np.asarray(scores, dtype=float)\n    positives = int(np.nansum(labels == 1))\n    negatives = int(np.nansum(labels == 0))\n    if not negatives or (minimum_positives is not None and positives < minimum_positives):\n        # Without usable labels the budget cannot be measured; fall back to the\n        # large, label-free training distribution rather than guessing.\n        chosen, status = float(fallback), f\'support fallback: {positives} positives, {negatives} negatives\'\n    else:\n        chosen = false_positive_floor(labels, scores, max_false_positive_rate)\n        status = f\'within {max_false_positive_rate:.3%} calibration false-positive budget\'\n    known = np.isfinite(labels)\n    realised = float(np.mean(scores[known & (labels == 0)] >= chosen)) if negatives else float(\'nan\')\n    return dict(threshold=chosen, status=status, calibration_false_positive_rate=realised,\n                calibration_alert_rate=float(np.mean(scores >= chosen)) if len(scores) else float(\'nan\'),\n                calibration_positives=positives)\n\ndef event_contained(a, mask):\n    total = a.loc[a.event_id.notna()].groupby(\'event_id\').size()\n    count = a.loc[mask & a.event_id.notna()].groupby(\'event_id\').size()\n    eligible = count.index[count.eq(total.reindex(count.index))]\n    return mask & (a.event_id.isna() | a.event_id.isin(eligible))\n\ndef forest_score(fitted, x):\n    return -fitted[\'model\'].score_samples(np.log1p(x[fitted[\'columns\']]))\n\ndef fit_forest(spec, x, settings):\n    columns = rhythm_views(x)[spec[\'view\']]\n    model = IsolationForest(n_estimators=settings[\'trees\'], max_samples=min(spec[\'samples\'],len(x)),\n                            contamination=\'auto\', random_state=settings[\'seed\'], n_jobs=2).fit(np.log1p(x[columns]))\n    return dict(model=model, columns=columns, spec=spec)\n\ndef calibrate_forest(fitted, a, x, budget):\n    score = forest_score(fitted, x)\n    threshold = fpr_capped_threshold(a.label, score, float(\'nan\'), budget)\n    if a.label.nunique() != 2 or not np.isfinite(threshold[\'threshold\']):\n        raise ValueError(\'Calibration needs explicit normal and anomaly labels; no unsupported fallback\')\n    return dict(threshold, support_warning=int(a.label.eq(1).sum()) < 30)\n\n"""Causal companion experiment inspired by BenjaminKakai\'s Spendly PR #3.\n\nIndependent implementation: completed-history baselines, no test-label sweep,\nsame-merchant/category duplicate windows, and calibrated alert budgets.\nThese research scores describe unusual spending, not fraud or confirmed duplicates.\n"""\nfrom collections import deque\n\nimport numpy as np\nimport pandas as pd\n\n\nCOLLECTIVE_PROTOCOL = dict(\n    source=\'https://github.com/Eeshan-Vaghjiani/Spendly/pull/3\',\n    baseline_days=90, warmup_days=28, minimum_weekend_days=4,\n    duplicate_hours=3, duplicate_tolerance=.02,\n    frequency_divisor=4., weekend_divisor=6.,\n    rules_budget=.01, combined_component_budget=.005,\n    selection=\'diagnostic comparator only; retain the V6 single forest\',\n)\n\n\ndef collective_features(data):\n    """Score each timestamp batch before updating history; return original row order.\n\n    The current transaction contributes to its own day-to-date total; simultaneous\n    peers do not see each other. Completed observed zero days enter rate baselines.\n    Matching duplicates must have a nonempty merchant and category, identical for\n    both transactions. No labels, event IDs or future totals inform these features.\n    """\n    required={\'user_id\',\'transaction_id\',\'transaction_timestamp\',\'amount\',\'merchant\',\n              \'category\',\'observation_start\'}\n    if not required <= set(data):\n        raise ValueError(\'Missing collective feature columns\')\n    if data.transaction_id.duplicated().any():\n        raise ValueError(\'Transaction IDs must be unique\')\n    amounts=pd.to_numeric(data.amount,errors=\'coerce\')\n    if not np.isfinite(amounts).all() or (amounts<=0).any():\n        raise ValueError(\'Expected finite positive expense amounts\')\n    work=data.copy().reset_index(drop=True)\n    work[\'_position\']=np.arange(len(work))\n    work[\'transaction_timestamp\']=pd.to_datetime(work.transaction_timestamp)\n    work[\'observation_start\']=pd.to_datetime(work.observation_start)\n    results=[]\n    for _,group in work.groupby(\'user_id\',sort=False):\n        if group.observation_start.nunique()!=1:\n            raise ValueError(\'Inconsistent observation start\')\n        start=group.observation_start.iloc[0]\n        if (group.transaction_timestamp<start).any():\n            raise ValueError(\'Transaction precedes observation\')\n        day=start.normalize()\n        counts,weekends,recent=deque(maxlen=90),deque(maxlen=26),deque()\n        day_count=0; day_amount=0.\n        for stamp,batch in group.groupby(\'transaction_timestamp\',sort=True):\n            current=stamp.normalize()\n            while day<current:\n                # Partial observation-start day is not a complete baseline day.\n                if day>=start:\n                    counts.append(day_count)\n                    if day.weekday()>=5:\n                        weekends.append(day_amount)\n                day+=pd.Timedelta(days=1); day_count=0; day_amount=0.\n            while recent and recent[0][0]<stamp-pd.Timedelta(hours=3):\n                recent.popleft()\n            eligible=len(counts)>=28\n            rate=max(float(np.mean(counts)),1.) if counts else 1.\n            weekend_base=max(float(np.median(weekends)),1.) if weekends else 1.\n            for row in batch.to_dict(\'records\'):\n                merchant=row[\'merchant\']; category=row[\'category\']; amount=float(row[\'amount\'])\n                known=(pd.notna(merchant) and pd.notna(category)\n                       and bool(str(merchant).strip()) and bool(str(category).strip()))\n                matches=sum(1 for _,a,m,c in recent if known and m==merchant and c==category\n                            and abs(a-amount)/max(a,amount)<=.02)\n                frequency=(day_count+1)/rate if eligible else 0.\n                weekend=((day_amount+amount)/weekend_base\n                         if eligible and current.weekday()>=5 and len(weekends)>=4 else 0.)\n                duplicate=float(matches) if eligible else 0.\n                results.append((row[\'_position\'],frequency,weekend,duplicate,eligible,\n                                max(frequency/4.,weekend/6.,duplicate)))\n            for row in batch.to_dict(\'records\'):\n                recent.append((stamp,float(row[\'amount\']),row[\'merchant\'],row[\'category\']))\n                day_count+=1; day_amount+=float(row[\'amount\'])\n    columns=[\'position\',\'day_count_ratio\',\'weekend_ratio\',\'duplicate_count\',\'eligible\',\'collective_score\']\n    out=pd.DataFrame(results,columns=columns).sort_values(\'position\').drop(columns=\'position\')\n    out.index=data.index\n    return out\n\n\ndef capped_rule_threshold(scores, labels, budget):\n    """Tie-safe empirical negative FPR cap, fitted only on calibration labels.\n\n    Positive labels never optimize the threshold. Unknown labels are an error.\n    No negatives means disabled (infinity), rather than uncalibrated alerts.\n    """\n    scores=np.asarray(scores,dtype=float); labels=np.asarray(labels,dtype=float)\n    if scores.shape!=labels.shape or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid calibration arrays\')\n    if not np.isin(labels,[0.,1.]).all() or not 0<budget<1:\n        raise ValueError(\'Explicit binary labels and a fractional budget are required\')\n    negatives=np.sort(scores[labels==0])\n    if not len(negatives):\n        return dict(threshold=float(\'inf\'),negative_support=0,budget=budget,\n                    calibration_FPR=None,status=\'disabled: no labelled negatives\')\n    allowed=int(np.floor(budget*len(negatives)))\n    threshold=max(1.,float(np.nextafter(negatives[len(negatives)-allowed-1],np.inf)))\n    return dict(threshold=threshold,negative_support=len(negatives),budget=budget,\n                calibration_FPR=float(np.mean(negatives>=threshold)),\n                status=\'calibrated\' if allowed else \'calibrated: fewer than one allowed false positive\')\n\n\ndef collective_diagnostics(rows, flags, model):\n    """Descriptive family/event coverage and daily alert burden; never selection."""\n    frame=rows.reset_index(drop=True).copy()\n    flags=np.asarray(flags,dtype=bool)\n    if len(flags)!=len(frame):\n        raise ValueError(\'Prediction alignment mismatch\')\n    frame[\'flag\']=flags\n    frame[\'day\']=pd.to_datetime(frame.time).dt.normalize()\n    records=[]\n    positives=frame.loc[frame.label.eq(1)]\n    for family,part in positives.groupby(\'family\',dropna=False):\n        events=part.dropna(subset=[\'event_id\']).groupby([\'user_id\',\'event_id\']).flag.any()\n        records.append(dict(model=model,family=family,positive_transactions=len(part),\n            transaction_recall=float(part.flag.mean()),events_with_ids=len(events),\n            event_recall=float(events.mean()) if len(events) else None))\n    daily=frame.groupby([\'user_id\',\'day\']).flag.any()\n    return records,dict(model=model,transactions=len(frame),alerts=int(flags.sum()),\n        active_user_days=len(daily),alerted_user_days=int(daily.sum()),\n        fraction_active_days_alerted=float(daily.mean()) if len(daily) else None,\n        event_definition=\'at least one flagged labelled member; absent event IDs excluded\',\n        daily_denominator=\'observed days containing evaluated transactions, not all calendar days\')\n'
MODULE_DIR=WORK_ROOT/'spendly_alert_review_modules'
MODULE_DIR.mkdir(exist_ok=True)
if str(MODULE_DIR) not in sys.path: sys.path.insert(0,str(MODULE_DIR))
WORKER_NAME='spendly_alert_worker_'+CODE_HASH[:12]
IF_NAME='spendly_alert_if_'+CODE_HASH[:12]
for name,source in [(WORKER_NAME,WORKER_SOURCE),(IF_NAME,IF_SOURCE)]:
    (MODULE_DIR/(name+'.py')).write_text(source,encoding='utf-8')
WORKER=importlib.import_module(WORKER_NAME)
IF_REFERENCE=importlib.import_module(IF_NAME)


In [ ]:
DATASET=load_manifest(INPUT_ROOT)
if DATASET['version']!='spendly-synthetic-r3-v1':
    raise ValueError('Use original R3 for this reproduction, not a different dataset')
versions={n:importlib.metadata.version(n) for n in ('numpy','pandas','scikit-learn','joblib')}
identity=digest(dict(code=CODE_HASH,inputs={k:v['sha256'] for k,v in DATASET['cohorts'].items()},
    manifest=DATASET,mode='software_smoke' if SMOKE else 'full_development',versions=versions))
bind_run(RESULTS,identity)
lock=dict(protocol='v7-alert-reproduction-v1',code_hash=CODE_HASH,run_identity=identity,
    model='frozen V7 single IF plus diagnostic comparators',scope='reproduction, not recovered historical results',
    inputs={k:v['sha256'] for k,v in DATASET['cohorts'].items()},versions=versions,
    manifest=DATASET,
    execution_mode='software_smoke' if SMOKE else 'full_development')
lock_path=RESULTS/'configuration_lock.json'
if lock_path.exists():
    if json.loads(lock_path.read_text())!=lock: raise ValueError('Existing alert configuration differs')
else: atomic_json(lock_path,lock)
TRAIN_RAW=load_cohort('train',INPUT_ROOT,DATASET,WORKER,RESULTS)
END=TRAIN_RAW.observation_start.min()+pd.Timedelta(weeks=int(DATASET['weeks_per_user']*.77))
TRAIN=restrict_history(TRAIN_RAW,END)
split_record=dict(training_start=str(TRAIN.observation_start.min()),training_end=str(END),
    evaluation_start=str(END),weeks_per_user=DATASET['weeks_per_user'],run_identity=identity)
split_path=RESULTS/'data_split_manifest.json'
if split_path.exists():
    if json.loads(split_path.read_text())!=split_record: raise ValueError('Saved evaluation boundary differs')
else: atomic_json(split_path,split_record)
del TRAIN_RAW
CALIBRATION=load_cohort('calibration',INPUT_ROOT,DATASET,WORKER,RESULTS)
VALIDATION=load_cohort('validation',INPUT_ROOT,DATASET,WORKER,RESULTS)
if set(TRAIN.user_id)&set(VALIDATION.user_id): raise ValueError('Train/validation users overlap')
RUNNER=SimpleNamespace(data=TRAIN,worker=WORKER,directory=RESULTS,workers=FEATURE_WORKERS,
    smoke=SMOKE,code_hash=CODE_HASH,identity=identity,benchmark_role=DATASET['benchmark_role'])


In [ ]:
OUTPUTS=['anomaly_metrics.csv','collective_protocol.json','anomaly_per_type_events.csv',
    'anomaly_alert_burden.json','isolation_forests.joblib','anomaly_protocol.json']
checkpoint_stage(RUNNER,'anomalies',lambda:anomaly_pipeline(RUNNER,CALIBRATION,VALIDATION,IF_REFERENCE),
    OUTPUTS,inputs=lock['inputs'])
print(pd.read_csv(RESULTS/'anomaly_metrics.csv').to_string(index=False))
print('Per-type and event results:')
print(pd.read_csv(RESULTS/'anomaly_per_type_events.csv').to_string(index=False))
write_json(RESULTS/'input_access_log.json',ACCESSES)
timing_path=RESULTS/('resume_timings_'+str(time.time_ns())+'.json') if (RESULTS/'timings.json').exists() else RESULTS/'timings.json'
write_json(timing_path,dict(seconds=dict(TIMINGS),scope='current process only; resumed completed stage is skipped'))
write_json(RESULTS/'artifact_manifest.json',dict(protocol='v7-alert-reproduction-v1',code_hash=CODE_HASH,
    files={p.relative_to(RESULTS).as_posix():recovery_hash(p) for p in RESULTS.rglob('*')
        if p.is_file() and p.name!='artifact_manifest.json'},
    execution_mode=lock['execution_mode'],evidence_eligible=not SMOKE,
    forecast_training=False,holdouts_accessed=False))
ARCHIVE=backup_run(RESULTS,'results')
print('ALERT REVIEW COMPLETE. Download this ZIP:',ARCHIVE)
